# VERONICA — wake word training (Colab, v2)

Trains `veronica_wake_model.onnx` for VERONICA with the official openWakeWord pipeline.

**Why v2:** the previous notebook trained without any real "not Veronica" audio
(`feature_data_files: {}`), so the model learned to say "Veronica" for *everything* —
it scored 0.999 on pure silence. v2 adds what openWakeWord's own recipe uses:

| Data | What it teaches the model |
|---|---|
| 2,000 h of pre-computed speech/noise/music features (ACAV100M) | what is **not** "Veronica" |
| AudioSet + FMA music clips | real background sounds mixed into training clips |
| MIT room impulse responses | how voices echo in real rooms |
| Synthetic "Veronica" clips (Piper TTS, many voices) | what **is** "Veronica" |
| Similar-sounding words (Verona, Monica, …) | not to wake on near-misses |

## How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Keep this tab open. Total time is roughly **1.5–3 hours**.
4. The last cell downloads **one file**: `veronica_wake_model.onnx` (about 200 KB).
5. Put it in `Veronica\veronica\models\` and **delete** any `veronica_wake_model.onnx.data` there.

If a cell turns red, copy the error text and send it to Claude. Every cell can be re-run;
finished steps are skipped automatically.

In [ ]:
# 1. Settings and helpers  (the defaults are good; edit only if you want to)
TARGET_PHRASE = "Veronica"
MODEL_NAME = "veronica_wake_model"
N_SAMPLES = 10000                 # synthetic "Veronica" training clips (more = better, slower)
N_SAMPLES_VAL = 1000              # synthetic clips held back for testing
TRAIN_STEPS = 20000
FALSE_ACTIVATION_PENALTY = 1500   # higher = fewer false wake-ups, may miss a few real ones
CUSTOM_NEGATIVES = [              # near-miss words the model must NOT wake on
    "verona", "veronique", "monica", "harmonica", "electronica",
    "ronica", "veranda", "very nice", "erica", "america",
]
SAVE_TO_DRIVE = False             # True = also copy the final model to your Google Drive
USE_MY_VOICE = True               # cell 5b asks for my_voice.zip (from record_wake_samples.py)

import os, sys, json, subprocess, textwrap

WORK = "/content/veronica_wake"
VENV = "/content/veronica-venv"
VENV_PY = f"{VENV}/bin/python"
os.makedirs(WORK, exist_ok=True)

with open(f"{WORK}/settings.json", "w") as f:
    json.dump({
        "work": WORK, "target_phrase": TARGET_PHRASE, "model_name": MODEL_NAME,
        "n_samples": N_SAMPLES, "n_samples_val": N_SAMPLES_VAL, "steps": TRAIN_STEPS,
        "max_negative_weight": FALSE_ACTIVATION_PENALTY, "custom_negatives": CUSTOM_NEGATIVES,
    }, f, indent=1)


def run(cmd, cwd="/content"):
    """Run a command, stream its output live, return (exit code, last ~8000 chars of output)."""
    # MPLBACKEND: Colab sets an inline plot backend the training env cannot load.
    env = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = ""
    while True:
        chunk = os.read(p.stdout.fileno(), 4096)
        if not chunk:
            break
        text = chunk.decode("utf-8", "replace")
        print(text, end="", flush=True)
        tail = (tail + text)[-8000:]
    return p.wait(), tail


def run_py(code, name="step"):
    """Run Python code inside the isolated Python 3.11 training environment."""
    path = f"{WORK}/_{name}.py"
    with open(path, "w") as f:
        f.write(textwrap.dedent(code))
    rc, _ = run([VENV_PY, path])
    if rc != 0:
        raise RuntimeError(f"Step '{name}' failed (exit {rc}). The real error is printed above.")


print("PASS: settings saved to", f"{WORK}/settings.json")

In [ ]:
# 2. GPU check + isolated Python 3.11 environment with pinned dependencies (~5 min)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
print("GPU:", gpu.stdout.strip())

# Colab's own Python changes over time; training runs in a fixed Python 3.11
# environment so these versions (the set openWakeWord's training was built on) keep working.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
if not os.path.exists(VENV_PY):
    subprocess.check_call(["uv", "venv", "--python", "3.11", VENV])

PACKAGES = [
    # core
    "torch==2.1.2", "torchaudio==2.1.2",          # PyPI wheels bundle CUDA 12.1 (works on Colab GPUs)
    "numpy==1.26.4", "scipy==1.11.4", "scikit-learn==1.3.2",
    "onnx==1.15.0", "onnxruntime==1.16.3",
    # openWakeWord training stack
    "torchinfo==1.8.0", "torchmetrics==1.2.0",
    "speechbrain==0.5.14", "audiomentations==0.33.0", "torch-audiomentations==0.11.0",
    "acoustics==0.2.6", "mutagen==1.47.0", "pronouncing==0.2.0", "deep-phonemizer==0.0.19",
    # Piper synthetic-voice generator
    "piper-phonemize==1.1.0", "webrtcvad-wheels==2.0.14",
    # datasets / audio I/O
    "datasets==2.14.6", "huggingface_hub==0.19.4", "pyarrow==14.0.2",
    "soundfile==0.12.1", "librosa==0.10.1", "numba==0.59.1",
    # misc
    "pyyaml", "tqdm", "requests",
    "setuptools==69.5.1",                         # provides pkg_resources (used by pronouncing); newer setuptools drops it
]
rc, _ = run(["uv", "pip", "install", "--python", VENV_PY, *PACKAGES])
if rc != 0:
    raise RuntimeError("Dependency install failed - see the error above.")

run_py("""
    import sys, numpy, scipy, torch, torchaudio, onnx, onnxruntime, datasets, sklearn
    import speechbrain, audiomentations, torch_audiomentations, acoustics, mutagen
    import torchinfo, torchmetrics, pronouncing, piper_phonemize, webrtcvad, soundfile, librosa
    print("python      ", sys.version.split()[0])
    print("torch       ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    print("numpy       ", numpy.__version__)
    print("onnx        ", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
    print("datasets    ", datasets.__version__)
    if not torch.cuda.is_available():
        raise SystemExit("FAIL: torch cannot see the GPU")
    print("PASS: all training dependencies import")
""", "deps")

In [ ]:
# 3. Source code: Piper sample generator + official openWakeWord (~3 min)
PSG_DIR = "/content/piper-sample-generator"
OWW_DIR = "/content/openwakeword"

if not os.path.isdir(f"{PSG_DIR}/.git"):
    subprocess.run(["rm", "-rf", PSG_DIR], check=True)
    subprocess.check_call(["git", "clone", "-q", "https://github.com/rhasspy/piper-sample-generator", PSG_DIR])

# openWakeWord's train.py was written for Piper sample generator v2.0.0: a flat
# generate_samples.py that runs the .pt voice model itself. Newer revisions import
# the separate `piper` package (from piper import PiperVoice) and break training.
def piper_is_v2():
    p = f"{PSG_DIR}/generate_samples.py"
    return os.path.isfile(p) and "from piper import" not in open(p).read()

if not piper_is_v2():
    subprocess.run(["git", "-C", PSG_DIR, "fetch", "-q", "--tags"], check=True)
    history = subprocess.run(["git", "-C", PSG_DIR, "log", "--format=%H", "--", "generate_samples.py"],
                             text=True, capture_output=True).stdout.split()
    for ref in ["v2.0.0", *history]:
        r = subprocess.run(["git", "-C", PSG_DIR, "show", f"{ref}:generate_samples.py"],
                           text=True, capture_output=True)
        if r.returncode == 0 and "from piper import" not in r.stdout:
            subprocess.check_call(["git", "-C", PSG_DIR, "checkout", "-q", "-f", ref])
            break
if not piper_is_v2():
    raise RuntimeError("Could not find the Piper v2.0.0 generator - the notebook will not guess.")
print("PASS: Piper revision", subprocess.check_output(
    ["git", "-C", PSG_DIR, "describe", "--tags", "--always"], text=True).strip())

PIPER_MODEL = f"{PSG_DIR}/models/en_US-libritts_r-medium.pt"
os.makedirs(os.path.dirname(PIPER_MODEL), exist_ok=True)
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    subprocess.check_call(["wget", "-q", "-c", "--tries=5", "--timeout=300", "-O", PIPER_MODEL,
        "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"])
if os.path.getsize(PIPER_MODEL) < 100_000_000:
    raise RuntimeError("Piper voice model download is incomplete - re-run this cell.")
print(f"PASS: Piper voice model {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB")

if not os.path.isfile(f"{OWW_DIR}/openwakeword/train.py"):
    subprocess.run(["rm", "-rf", OWW_DIR], check=True)
    subprocess.check_call(["git", "clone", "-q", "https://github.com/dscripka/openWakeWord", OWW_DIR])
# --no-deps: its dependencies are already pinned in cell 2.
subprocess.check_call(["uv", "pip", "install", "-q", "--python", VENV_PY, "--no-deps", "-e", OWW_DIR])

# Shared feature models (melspectrogram + speech embedding) that every openWakeWord model uses.
run_py("""
    import os, openwakeword, openwakeword.utils as u
    u.download_models()
    d = os.path.join(os.path.dirname(openwakeword.__file__), "resources", "models")
    for n in ("melspectrogram.onnx", "embedding_model.onnx"):
        p = os.path.join(d, n)
        if not (os.path.isfile(p) and os.path.getsize(p) > 100_000):
            raise SystemExit("FAIL: missing " + p)
        print("PASS:", n)
""", "oww_models")

In [ ]:
# 4. Training data: room echoes, background audio, and the big negative feature set (~20-40 min)
#    Needs ~20 GB of Colab disk. Already-downloaded parts are skipped on re-run.
run_py("""
    import os, glob, json, subprocess, tarfile
    import numpy as np, scipy.io.wavfile, datasets
    from tqdm import tqdm

    S = json.load(open("/content/veronica_wake/settings.json"))
    W = S["work"]

    def write16(path, arr):
        arr = np.asarray(arr, dtype=np.float32)
        scipy.io.wavfile.write(path, 16000, (np.clip(arr, -1, 1) * 32767).astype(np.int16))

    def count(d):
        return len(glob.glob(d + "/*.wav"))

    # a) MIT room impulse responses (reverb)
    rir = f"{W}/mit_rirs"; os.makedirs(rir, exist_ok=True)
    if count(rir) < 200:
        ds = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
        ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
        for row in tqdm(ds, desc="room echoes"):
            write16(f"{rir}/" + os.path.basename(row["audio"]["path"]).rsplit(".", 1)[0] + ".wav", row["audio"]["array"])
    print("room echo files:", count(rir))
    if count(rir) == 0:
        raise SystemExit("FAIL: no room impulse responses downloaded")

    # b) AudioSet background sounds (one ~2,000-clip shard)
    aud = f"{W}/audioset_16k"; os.makedirs(aud, exist_ok=True)
    if count(aud) < 500:
        try:
            tar = f"{W}/bal_train09.tar"
            subprocess.run(["wget", "-q", "-c", "--tries=5", "-O", tar,
                "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar"], check=True)
            with tarfile.open(tar) as t:
                t.extractall(f"{W}/audioset_raw")
            files = [str(p) for p in glob.glob(f"{W}/audioset_raw/**/*.flac", recursive=True)]
            ds = datasets.Dataset.from_dict({"audio": files}).cast_column("audio", datasets.Audio(sampling_rate=16000))
            for row in tqdm(ds, desc="AudioSet"):
                write16(f"{aud}/" + os.path.basename(row["audio"]["path"]).replace(".flac", ".wav"), row["audio"]["array"])
        except Exception as e:
            print("WARN: AudioSet download failed:", type(e).__name__, e)
    print("AudioSet files:", count(aud))

    # c) FMA music (1 hour)
    fma = f"{W}/fma"; os.makedirs(fma, exist_ok=True)
    if count(fma) < 100:
        try:
            ds = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
            it = iter(ds.cast_column("audio", datasets.Audio(sampling_rate=16000)))
            for i in tqdm(range(3600 // 30), desc="FMA music"):
                row = next(it)
                write16(f"{fma}/" + os.path.basename(row["audio"]["path"]).rsplit(".", 1)[0] + ".wav", row["audio"]["array"])
        except Exception as e:
            print("WARN: FMA download failed:", type(e).__name__, e)
    print("FMA files:", count(fma))

    # d) Last-resort fallback so augmentation still has *some* background.
    if count(aud) + count(fma) < 100:
        noise = f"{W}/noise_16k"; os.makedirs(noise, exist_ok=True)
        rng = np.random.default_rng(0)
        for i in range(300):
            spec = np.fft.rfft(rng.normal(size=160000))
            f = np.arange(1, spec.size + 1)
            spec /= f ** (0.5 * (i % 3))   # white / pink / brown noise
            x = np.fft.irfft(spec); x = 0.3 * x / (np.abs(x).max() + 1e-9)
            write16(f"{noise}/noise_{i:03d}.wav", x)
        print("WARN: using generated noise as background (real downloads failed). Quality will be lower.")

    # e) Pre-computed negative features (the part the old notebook was missing)
    base = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/"
    for name, min_bytes in (("openwakeword_features_ACAV100M_2000_hrs_16bit.npy", 1_000_000_000),
                            ("validation_set_features.npy", 10_000_000)):
        path = f"{W}/{name}"
        if not os.path.isfile(path) or os.path.getsize(path) < min_bytes:
            print("downloading", name, "...")
            subprocess.run(["wget", "-q", "-c", "--tries=10", "--timeout=300", "-O", path, base + name], check=True)
        arr = np.load(path, mmap_mode="r")
        print(f"PASS: {name}: shape={arr.shape} ({os.path.getsize(path)/1e9:.2f} GB)")

    subprocess.run(["df", "-h", "/content"])
    print("PASS: training data ready")
""", "data")

In [ ]:
# 5. Training config + synthetic "Veronica" / near-miss clips (~20-40 min on T4)
run_py("""
    import os, json, glob, yaml
    S = json.load(open("/content/veronica_wake/settings.json"))
    W = S["work"]

    # Start from openWakeWord's own example config, then set every key train.py needs.
    base = "/content/openwakeword/examples/custom_model.yml"
    cfg = yaml.safe_load(open(base)) if os.path.isfile(base) else {}
    bg = [d for d in (f"{W}/audioset_16k", f"{W}/fma", f"{W}/noise_16k") if glob.glob(d + "/*.wav")]
    cfg.update({
        "model_name": S["model_name"],
        "target_phrase": [S["target_phrase"]],
        "custom_negative_phrases": S["custom_negatives"],
        "n_samples": S["n_samples"],
        "n_samples_val": S["n_samples_val"],
        "tts_batch_size": 50,
        "augmentation_batch_size": 16,
        "piper_sample_generator_path": "/content/piper-sample-generator",
        "output_dir": f"{W}/output",
        "rir_paths": [f"{W}/mit_rirs"],
        "background_paths": bg,
        "background_paths_duplication_rate": [1] * len(bg),
        "augmentation_rounds": 1,
        "false_positive_validation_data_path": f"{W}/validation_set_features.npy",
        "feature_data_files": {"ACAV100M_sample": f"{W}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
        "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
        "model_type": "dnn",
        "layer_size": 32,
        "steps": S["steps"],
        "max_negative_weight": S["max_negative_weight"],
        "target_false_positives_per_hour": 0.2,
        "target_accuracy": 0.5,
        "target_recall": 0.25,
    })
    os.makedirs(cfg["output_dir"], exist_ok=True)
    yaml.safe_dump(cfg, open(f"{W}/my_model.yaml", "w"), sort_keys=False)
    print(open(f"{W}/my_model.yaml").read())
    print("PASS: config written")
""", "config")

rc, _ = run([VENV_PY, f"{OWW_DIR}/openwakeword/train.py", "--training_config", f"{WORK}/my_model.yaml", "--generate_clips"])
if rc != 0:
    raise RuntimeError("Clip generation failed - see the error above.")

clips = f"{WORK}/output/{MODEL_NAME}"
for d in ("positive_train", "positive_test", "negative_train", "negative_test"):
    n = len([x for x in os.listdir(f"{clips}/{d}") if x.endswith(".wav")]) if os.path.isdir(f"{clips}/{d}") else 0
    print(f"{d:15s}: {n} clips")
    if n == 0:
        raise RuntimeError(f"No clips were generated in {d}.")
print("PASS: synthetic clips generated")

In [ ]:
# 5b. Add YOUR voice: upload my_voice.zip (made on your PC by record_wake_samples.py)
#     The synthetic voices are American English; your own clips teach the model your accent.
import glob, random, shutil, zipfile
VOICE = f"{WORK}/my_voice"
clips = f"{WORK}/output/{MODEL_NAME}"

if not globals().get("USE_MY_VOICE", True):
    print("SKIP: USE_MY_VOICE is False")
else:
    if not glob.glob(f"{VOICE}/positive/*.wav"):
        from google.colab import files
        print("Click 'Choose files' and pick  Veronica\\veronica\\training_voice\\my_voice.zip")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded - run this cell again and choose my_voice.zip.")
        with zipfile.ZipFile(next(iter(uploaded))) as z:
            z.extractall(VOICE)

    pos = sorted(glob.glob(f"{VOICE}/positive/*.wav"))
    neg = sorted(glob.glob(f"{VOICE}/negative/*.wav"))
    if len(pos) < 10:
        raise RuntimeError(f"Only {len(pos)} 'Veronica' clips in the zip - record at least 10.")

    # Hold back ~20% of your "Veronica" clips so cell 8 can test on real voice it never trained on.
    random.Random(0).shuffle(pos)
    n_test = max(4, len(pos) // 5)
    test, train = pos[:n_test], pos[n_test:]
    real_test = f"{WORK}/real_voice_test"
    shutil.rmtree(real_test, ignore_errors=True)
    os.makedirs(real_test)
    for p in test:
        shutil.copy2(p, real_test)

    # Each copy gets different random echo/background in cell 6, so repeats are not duplicates.
    REPEAT_POS, REPEAT_NEG = 50, 20
    for d in ("positive_train", "negative_train"):
        for old in glob.glob(f"{clips}/{d}/me_*.wav"):   # re-running must not double up
            os.remove(old)
    for src_list, d, reps in ((train, "positive_train", REPEAT_POS), (neg, "negative_train", REPEAT_NEG)):
        for p in src_list:
            stem = os.path.basename(p)[:-4]
            for k in range(reps):
                shutil.copy2(p, f"{clips}/{d}/{stem}_r{k:02d}.wav")

    print(f"your 'Veronica' clips : {len(train)} for training (x{REPEAT_POS}), {len(test)} held back for testing")
    print(f"your other phrases    : {len(neg)} (x{REPEAT_NEG}) as things NOT to wake on")
    print("PASS: your voice added - now run cell 6, 7, 8, 9")

In [ ]:
# 6. Augmentation (echo + background mixing) and feature extraction (~20-60 min)
rc, _ = run([VENV_PY, f"{OWW_DIR}/openwakeword/train.py", "--training_config", f"{WORK}/my_model.yaml", "--augment_clips",
            "--overwrite"])   # rebuild so clips added in 5b are included
if rc != 0:
    raise RuntimeError("Augmentation / feature extraction failed - see the error above.")

for n in ("positive_features_train.npy", "negative_features_train.npy",
          "positive_features_test.npy", "negative_features_test.npy"):
    p = f"{WORK}/output/{MODEL_NAME}/{n}"
    if not os.path.isfile(p) or os.path.getsize(p) == 0:
        raise RuntimeError(f"Missing feature file: {p}")
    print(f"PASS: {n} ({os.path.getsize(p)/1e6:.1f} MB)")

In [ ]:
# 7. Train the model (~15-30 min on T4)
import time
onnx_path = f"{WORK}/output/{MODEL_NAME}.onnx"
start = time.time()
rc, tail = run([VENV_PY, f"{OWW_DIR}/openwakeword/train.py", "--training_config", f"{WORK}/my_model.yaml", "--train_model"])
fresh = os.path.isfile(onnx_path) and os.path.getmtime(onnx_path) >= start

if rc != 0:
    # train.py converts to TFLite *after* saving the ONNX model. VERONICA only uses ONNX,
    # so a failure in that last optional step is fine - anything else is a real failure.
    if fresh and any(k in tail.lower() for k in ("tflite", "onnx_tf", "tensorflow")):
        print("\nNOTE: only the optional TFLite conversion failed. The ONNX model VERONICA needs was saved.")
    else:
        raise RuntimeError("Training failed - see the error above.")
if not fresh:
    raise RuntimeError("Training finished but no new ONNX model was written.")
print("PASS: trained", onnx_path)

In [ ]:
# 8. Make ONE self-contained model file + quality check (must pass before you download)
run_py("""
    import os, glob, json
    import numpy as np, onnx, onnxruntime as ort
    from scipy.io import wavfile
    from scipy.signal import resample_poly
    from openwakeword.model import Model

    S = json.load(open("/content/veronica_wake/settings.json"))
    W, name = S["work"], S["model_name"]
    src = f"{W}/output/{name}.onnx"
    out_dir = "/content/veronica_export"
    os.makedirs(out_dir, exist_ok=True)
    for old in glob.glob(out_dir + "/*"):
        os.remove(old)
    final = f"{out_dir}/{name}.onnx"

    # Newer PyTorch may write weights to a separate .onnx.data file. Embed everything
    # into one file, then load it from a folder with no side files to prove it is complete.
    onnx.save_model(onnx.load(src), final, save_as_external_data=False)
    if os.path.exists(final + ".data"):
        raise SystemExit("FAIL: weights were not embedded")
    ort.InferenceSession(final, providers=["CPUExecutionProvider"])
    print(f"PASS: self-contained model {final} ({os.path.getsize(final):,} bytes)")

    model = Model(wakeword_models=[final], inference_framework="onnx")
    key = list(model.models)[0]
    SR = 16000

    def stream_max(pcm):
        model.reset()
        best = 0.0
        for i in range(0, len(pcm) - 1280 + 1, 1280):
            best = max(best, float(model.predict(pcm[i:i + 1280])[key]))
        return best

    # --- Non-speech must NOT wake (the old model failed exactly this) ---
    rng = np.random.default_rng(0)
    t = np.arange(SR * 5) / SR
    non_speech = {
        "silence (zeros)": np.zeros(SR * 5, np.int16),
        "quiet room noise": rng.normal(0, 60, SR * 5).astype(np.int16),
        "loud noise": rng.normal(0, 3000, SR * 5).astype(np.int16),
        "440 Hz tone": (3000 * np.sin(2 * np.pi * 440 * t)).astype(np.int16),
    }
    fired = []
    for label, pcm in non_speech.items():
        s = stream_max(pcm)
        print(f"  {label:18s} score {s:.3f}  {'FAIL' if s >= 0.5 else 'ok'}")
        if s >= 0.5:
            fired.append(label)

    # --- Held-out synthetic clips ---
    def clip_max(path):
        sr, a = wavfile.read(path)
        a = a.astype(np.float32)
        if a.ndim > 1:
            a = a.mean(axis=1)
        if sr != SR:
            a = resample_poly(a, SR, sr)
        pad = np.zeros(SR, np.float32)
        return stream_max(np.concatenate([pad, a, pad]).astype(np.int16))

    pos = sorted(glob.glob(f"{W}/output/{name}/positive_test/*.wav"))[:200]
    neg = sorted(glob.glob(f"{W}/output/{name}/negative_test/*.wav"))[:200]
    ps = np.array([clip_max(p) for p in pos])
    ns = np.array([clip_max(p) for p in neg])
    recall = float((ps >= 0.5).mean())
    near_miss = float((ns >= 0.5).mean())
    print(f"  'Veronica' clips detected : {recall:.0%} of {len(ps)}  (median score {np.median(ps):.2f})")
    print(f"  near-miss words that woke : {near_miss:.0%} of {len(ns)}")

    # Your own held-back recordings (from cell 5b) - the most honest test.
    real = sorted(glob.glob(f"{W}/real_voice_test/*.wav"))
    real_recall = None
    if real:
        rs = np.array([clip_max(p) for p in real])
        real_recall = float((rs >= 0.5).mean())
        print(f"  YOUR voice detected       : {real_recall:.0%} of {len(rs)}  (scores: "
              + " ".join(f"{x:.2f}" for x in rs) + ")")

    json.dump({"non_speech_fired": fired, "recall": recall, "near_miss_rate": near_miss,
               "real_voice_recall": real_recall,
               "bytes": os.path.getsize(final)}, open(f"{out_dir}/quality_report.json", "w"), indent=1)

    if fired:
        raise SystemExit("FAIL: the model wakes on non-speech: " + ", ".join(fired) + ". Do not use it.")
    if recall < 0.2:
        raise SystemExit("FAIL: the model almost never detects 'Veronica'. Do not use it.")
    if recall < 0.5:
        print("WARN: detection is on the low side; it may need you to speak clearly. Usable for testing.")
    if real_recall is not None and real_recall < 0.6:
        print("WARN: misses your real voice often - record more clips (vary distance/volume) and retrain.")
    if near_miss > 0.2:
        print("WARN: wakes on many similar-sounding words; consider raising FALSE_ACTIVATION_PENALTY.")
    print("PASS: quality check - safe to download")
""", "export_check")

In [ ]:
# 9. Download the model (one file)
import shutil
final = f"/content/veronica_export/{MODEL_NAME}.onnx"
if not os.path.isfile(final):
    raise RuntimeError("Run cell 8 first (it must print 'PASS: quality check').")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy2(final, f"/content/drive/MyDrive/{MODEL_NAME}.onnx")
    print("Copied to Google Drive: MyDrive/" + MODEL_NAME + ".onnx")

from google.colab import files
files.download(final)
print(f"Downloading {MODEL_NAME}.onnx ({os.path.getsize(final):,} bytes).")
print(r"Next: put it in Veronica\veronica\models\ , delete veronica_wake_model.onnx.data there,")
print("then run:  python voice_check.py --skip-stt")